In [0]:
# In this notebook we will train our geographic logistic regression models. 

# Fit separate logistic regression models for bike and dock stockout onset.
# The target is 1 if a new stockout occurs within the next 30 minutes, otherwise 0.
# Only currently non-stockout observations with sufficient future coverage are labeled.
#
# P(stockout) = 1 / (1 + exp(-eta))
#
# eta = intercept
#     + beta_1 * available
#     + beta_2 * available_fraction
#     + beta_3 * change_30min_fraction
#     + beta_4 * log_capacity
#     + beta_5 * weekend
#     + sum_k beta_k * harmonic_k
#     + geographic_intercept[cell]                                                  
#
# available is the current number of rentable bikes or available docks.
# available_fraction is available / station capacity.
# change_30min_fraction is the recent change per minute divided by capacity.
# log_capacity is ln(capacity), with nonpositive or missing capacity set to null.
# weekend is 1 on Saturday and Sunday, otherwise 0.
#
# The four harmonics are sin(2*pi*h/24), cos(2*pi*h/24),
# sin(4*pi*h/24), and cos(4*pi*h/24), where h is fractional New York hour.
# Geographic cells have separate intercept adjustments (one-hot encoded).          
# Every training cell has its own adjustment, unseen cells receive the overall baseline.  
#
# Numeric missing values are filled with training medians, then standardized. Missingness is 
# relatively low, about one tenth of one percent at maximum in the current gold dataset. It
# can arise when the station information is stale, at the beginning of training, or when there have been
# some missed data polls.

# Coefficients minimize training log loss plus an L2 (ridge) penalty.
# The penalty acts like a normal(0, tau^2) prior on each coefficient, so cells with
# little data are shrunk toward the overall baseline (partial pooling).
# In scikit-learn the penalty is set as C = tau^2. The Spark version used
# regParam = 1 / (n * tau^2) which implements the same prior.
# C is chosen by GridSearchCV, separately for bikes and docks.                 
#
# Cross-validation uses TimeGapSplit with an expanding window: the first fold     
# trains on 30 days, each fold validates on the next 7 days, and a 40-minute gap    
# separates training from validation, so training labels finish before validation begins.
#
# Free Edition version: the models are fitted with scikit-learn on a random sample of
# rows (SAMPLE_FRACTION), because the data must fit in memory. Logistic        
# regression has only a few hundred coefficients, so sampling changes the fit very little.
#
# MLflow records each run, and registers the best model with the "challenger" alias. 




from pyspark.sql import functions as F
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from datetime import timedelta
from sklego.model_selection import TimeGapSplit
from sklearn.model_selection import cross_validate, GridSearchCV

import mlflow.sklearn
from mlflow.models import infer_signature

import io
import mlflow
import pandas as pd
from mlflow import MlflowClient


# Scheduled jobs pass run_mode = "production". Interactive runs default to "dev".
# A dev run trains and records the run, but does not register the models.
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
RUN_MODE = dbutils.widgets.get("run_mode")

dbutils.widgets.text("schema", "workspace.default")
SCHEMA = dbutils.widgets.get("schema")

PREDICTOR_TABLE = f"{SCHEMA}.gold_station_predictors"
TARGET_TABLE = f"{SCHEMA}.gold_station_targets"

IS_PRODUCTION = RUN_MODE == "production"

# Dev runs go to a separate experiment, so trial runs do not mix with real ones.
# Adding this flag so that the notebook can be run in either free edition or main
# Note that models go into directories based on the user

CURRENT_USER = spark.sql("SELECT current_user()").first()[0]

EXPERIMENT_PATH = (
    f"/Users/{CURRENT_USER}/citibike-model-training"
    + ("" if IS_PRODUCTION else "-dev")
)

# We load the predictors
predictors = spark.table(PREDICTOR_TABLE)

# We load the targets for the 30-minute horizon
targets = spark.table(TARGET_TABLE).filter("horizon_minutes = 30")

# We put them together
full_data = predictors.join(targets,["station_id","fetched_at"])

# We list the numeric features. This is just to make the spark to pandas step more efficient

shared_features = ["log_capacity", "weekend", "hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2"]

numeric_features_bike = ["num_bikes_available", "bike_available_fraction",
                         "bike_change_30min_per_minute_fraction", *shared_features]
numeric_features_dock = ["num_docks_available", "dock_available_fraction",
                         "dock_change_30min_per_minute_fraction", *shared_features]
numeric_features = [*numeric_features_bike[:3], *numeric_features_dock[:3], *shared_features]

# To make sure this fits in memory, we will take a 10% sample of the training rows. I'll separately show
# a pyspark version of this

SAMPLE_FRACTION = 0.1

filtered_data = (full_data.filter(F.col("bike_target").isNotNull() | F.col("dock_target").isNotNull())
        .select(*numeric_features, "geographic_cell", "fetched_at",
                "bike_target", "dock_target")
        .sample(fraction=SAMPLE_FRACTION, seed=1).toPandas())


# Keeping the datatypes safe for pandas/mlflow

filtered_data[numeric_features] = filtered_data[numeric_features].astype("float64")


data_bike = filtered_data[filtered_data["bike_target"].notna()].reset_index(drop=True)
data_dock = filtered_data[filtered_data["dock_target"].notna()].reset_index(drop=True)

# Here is our sklearn pipeline

model_bike = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features_bike), # Median impute
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["geographic_cell"]), # One-Hot the geographic cells. Handle unknown lets us use the model as is on data which has a new value of the cells
        ]),
        LogisticRegression( max_iter=200), # Logistic regression, the convergence is slow with so many geo cells
    )

model_dock = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features_dock), # Median impute
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["geographic_cell"]), # One-Hot the geographic cells. Handle unknown lets us use the model as is on data which has a new value of the cells
        ]),
        LogisticRegression(C=0.25, max_iter=200), # Logistic regression
    )

# Now we configure cross-validation


# We are going to start out by trying an expanding window of 7 days


cv_bike = TimeGapSplit(date_serie=data_bike["fetched_at"], # Date
                  train_duration=timedelta(days=30), # Minimum training fold
                  valid_duration=timedelta(days=7), # Validation period
                  gap_duration=timedelta(minutes=40),  # Gap to prevent leakage
                  window="expanding" # Expanding window
                  )


# This will do some hyperparameter tuning
model_bike_cv_search = GridSearchCV(model_bike,
                      param_grid={"logisticregression__C": [0.04, 0.1,0.3, 1.0,3.0,10.0]}, 
                      cv=cv_bike,
                      scoring="neg_log_loss")




cv_dock = TimeGapSplit(date_serie=data_dock["fetched_at"], # Date
                  train_duration=timedelta(days=30), # Minimum training fold
                  valid_duration=timedelta(days=7), # Validation period
                  gap_duration=timedelta(minutes=40),  # Gap to prevent leakage
                  window="expanding" # Expanding window
                  )


# This will do some hyperparameter tuning
model_dock_cv_search = GridSearchCV(model_dock,
                      param_grid={"logisticregression__C": [0.04, 0.1,0.3, 1.0,3.0,10.0]}, 
                      cv=cv_dock,
                      scoring="neg_log_loss")




In [0]:
# Here we will train the bike model and save it using mlflow

mlflow.set_registry_uri("databricks-uc") # This will save the models in the databricks unity catalog

# Specialize to your use case
mlflow.set_experiment(EXPERIMENT_PATH)

model_name = f"{SCHEMA}.bike_stockout_lr"
input_columns = numeric_features_bike + ["geographic_cell"] 

# Begin the model run
with mlflow.start_run(run_name="bike_stockout_lr_gridsearch") as run:

    # We fit the model 
    model_bike_cv_search.fit(data_bike, data_bike["bike_target"])

    # We record the settings of this run
    mlflow.log_params({
        "C_grid": str(model_bike_cv_search.param_grid["logisticregression__C"]),
        "train_duration": str(cv_bike.train_duration),
        "valid_duration": str(cv_bike.valid_duration),
        "gap_duration": str(cv_bike.gap_duration),
        "window": cv_bike.window,
        "sample_fraction": SAMPLE_FRACTION,
    })
    mlflow.log_params(model_bike_cv_search.best_params_) # The hyperparameter chosen

    # Record the results. scikit-learn reports negative log loss, so we change the sign
    mlflow.log_metric("cv_log_loss", -model_bike_cv_search.best_score_)
    results = pd.DataFrame(model_bike_cv_search.cv_results_).drop(columns="params")
    mlflow.log_text(results.to_csv(index=False), "cv_results.csv") # The score of every C on every fold

    # Save the best model. GridSearchCV already refit it on all rows with the best C

    best_model = model_bike_cv_search.best_estimator_

    # The input_example gives an example of the data the model uses
    # and also is used to infer the model signature
    input_example = data_bike[input_columns].head(5)
    model_info = mlflow.sklearn.log_model(
        best_model, name="model",
        input_example=input_example,
        signature=infer_signature(input_example, best_model.predict_proba(input_example)),
        pyfunc_predict_fn="predict_proba",   # The saved model returns probabilities, we are principled
        serialization_format="skops",  # Safe choice for storing an sklearn model
        skops_trusted_types=["numpy.dtype"], # Cost of using skops here
        registered_model_name= model_name if IS_PRODUCTION else None)    #  Registers a new version of the model if prod

# The scoring notebook loads "champion model"
# We will for now save as challenger


if IS_PRODUCTION:
    MlflowClient().set_registered_model_alias(model_name, "challenger", model_info.registered_model_version)
    print("Registered version", model_info.registered_model_version)

print("Registered version", model_info.registered_model_version)

In [0]:
# Here we will train the dock model and save it using mlflow

mlflow.set_registry_uri("databricks-uc") # This will save the models in the databricks unity catalog

# Specialize to your use case
mlflow.set_experiment(EXPERIMENT_PATH)

model_name = f"{SCHEMA}.dock_stockout_lr"
input_columns = numeric_features_dock + ["geographic_cell"] 

# Begin the model run
with mlflow.start_run(run_name="dock_stockout_lr_gridsearch") as run:

    # We fit the model 
    model_dock_cv_search.fit(data_dock, data_dock["dock_target"])

    # We record the settings of this run
    mlflow.log_params({
        "C_grid": str(model_dock_cv_search.param_grid["logisticregression__C"]),
        "train_duration": str(cv_dock.train_duration),
        "valid_duration": str(cv_dock.valid_duration),
        "gap_duration": str(cv_dock.gap_duration),
        "window": cv_dock.window,
        "sample_fraction": SAMPLE_FRACTION,
    })
    mlflow.log_params(model_dock_cv_search.best_params_) # The hyperparameter chosen

    # Record the results. scikit-learn reports negative log loss, so we change the sign
    mlflow.log_metric("cv_log_loss", -model_dock_cv_search.best_score_)
    results = pd.DataFrame(model_dock_cv_search.cv_results_).drop(columns="params")
    mlflow.log_text(results.to_csv(index=False), "cv_results.csv") # The score of every C on every fold

    # Save the best model. GridSearchCV already refit it on all rows with the best C

    best_model = model_dock_cv_search.best_estimator_

    # The input_example gives an example of the data the model uses
    # and also is used to infer the model signature
    input_example = data_dock[input_columns].head(5)
    model_info = mlflow.sklearn.log_model(
        best_model, name="model",
        input_example=input_example,
        signature=infer_signature(input_example, best_model.predict_proba(input_example)),
        pyfunc_predict_fn="predict_proba",   # The saved model returns probabilities, we are principled
        serialization_format="skops",  # Safe choice for storing an sklearn model
        skops_trusted_types=["numpy.dtype"], # Cost of using skops here
        registered_model_name=model_name if IS_PRODUCTION else None)    #  Registers a new version of the model

# The scoring notebook loads "champion model"
# We will for now save as challenger if in production


if IS_PRODUCTION:
    MlflowClient().set_registered_model_alias(model_name, "challenger", model_info.registered_model_version)
    print("Registered version", model_info.registered_model_version)
print("Registered version", model_info.registered_model_version)